In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3 as sql

In [2]:
df = pd.read_csv('../gronagarden_data.csv')
df_clean = df.copy()

# 2. ETL-Pipeline
I denna notebook genomför vi datatvätt och berikning av rådatan från Gröna Gården. 
Målet är att transformera datan till ett format som är redo för KPI-analys och lagring i en SQL-databas.

### 2.1 Städat kolummnerna och rader
Datan innehöll stavfel, dubbletter, fel typer, saknade värden.
Det jag har gjort är:
* Slagit ihop dubbletter.
* Korrigerat felstavningar.
* Grupperat produkter i logiska kategorier.


In [3]:

def clean_siffra(num):
    # Gör allt till små bokstäver
    renad = num.astype(str).str.lower()
    
    # Översätter ord till siffror
    siffra_map = {'en': '1', 'ett': '1', 'två': '2', 'tre': '3', 'fyra': '4', 'fem': '5'}
    for ord, nr in siffra_map.items():
        #Går igenom varje ord i mappen
        renad = renad.str.replace(ord, nr)
    
    # Rensa bort allt utom siffror, punkt och komma
    renad = renad.str.replace(r'[^\d,.]', '', regex=True)
    
    # Fixa decimaler och gör till siffror
    return pd.to_numeric(renad.str.replace(',', '.'), errors='coerce').fillna(0)

# Använd på kolumnerna
# Här kallar du på den för dina huvudkolumner
for col in ['vikt_gram', 'pris', 'antal']:
    if col in df_clean.columns:
        df_clean[col] = clean_siffra(df_clean[col])

In [4]:

def clean_datum(datum):
    # Hantera tomma rutor (nullvärden)
    if pd.isna(datum) or str(datum).lower() == 'nan':
        return pd.NaT
    
    # Tvätta bort mellanslag och gör allt till små bokstäver
    d = str(datum).lower().strip()
    
    # Om datumet börjar på "20" (t.ex. 2024-01-04) tvingar vi formatet ÅÅÅÅ-MM-DD
    if d.startswith('20'):
        return pd.to_datetime(d, errors='coerce')

    # Översätt svenska månader till engelska
    manader = {
       'januari': 'january', 'februari': 'february', 'mars': 'march',
        'april': 'april', 'maj': 'may', 'juni': 'june',
        'juli': 'july', 'augusti': 'august', 'september': 'september',
        'oktober': 'october', 'november': 'november', 'december': 'december'
    }
    
    for sv, en in manader.items():
        if sv in d:
            d = d.replace(sv, en)
    
    # Omvandlar till datetime
    return pd.to_datetime(d, dayfirst=True, errors='coerce')

# --- 2. KÖR DATUMTVÄTTEN ---
for col in ['orderdatum', 'faktiskt_leveransdatum', 'recensionsdatum']:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].apply(clean_datum)


/var/folders/29/3kf793v13_j58pg81t38xr_m0000gn/T/ipykernel_13170/1774659378.py:11: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(d, errors='coerce')


In [5]:
def clean_product(namn):
    if pd.isna(namn):
        return "Okänd produkt"
    
    n = str(namn).lower().strip()
    
    # --- GRUPPERINGAR ---
    if 'basilika' in n or 'bassil' in n:
        return 'Basilika planta'
    
    # Samla ALL tomat-logik här
    if 'tom' in n: 
        if 'planta' in n:
            return 'Tomatplanta (olika sorter)'
        if 'biff' in n:
            return 'Bifftomat frön'
        if 'tiny' in n or 'tim' in n:
            return 'Tomat Tiny Tim frön'
        return 'Tomatfrön (standard)'

    if 'petunia' in n or 'petunior' in n:
        return 'Petunia mix'
        
    if 'narciss' in n or 'påsk' in n:
        return 'Påskliljor/Narcisser 15-pack'
        
    if 'tulpan' in n:
        return 'Tulpanlökar 20-pack'
        
    if 'vattenkanna' in n or 'vatten kanna' in n:
        return 'Vattenkanna 10L'
        
    if 'pelargon' in n:
        return 'Pelargon röd'
        
    if 'balkong' in n or 'plastlåda' in n:
        return 'Balkonglåda plast'
        
    if 'jord' in n:
        if '25' in n: return 'Blomjord 25L'
        if '50' in n: return 'Blomjord 50L'
        return 'Blomjord'
        
    if 'kruka' in n or 'terrakotta' in n: # Lade till terrakotta här!
        if '15' in n: return 'Lerkruka 15cm'
        if '25' in n: return 'Lerkruka 25cm'
        return 'Lerkruka'
        
    if 'sallat' in n or 'sallad' in n:
        return 'Sallad mix frön'
        
    if 'ört' in n or 'persilja' in n or 'dill' in n:
        return 'Örter mix'
        
    if 'monstera' in n:
        return 'Monstera'
        
    if 'sommar' in n or 'sommarblommor' in n:
        return 'Sommarblommor mix'
        
    if 'freds' in n:
        return 'Fredskalla'
        
    if 'gull' in n:
        return 'Gullranka'
        
    if 'gurk' in n:
        return 'Gurkfrön'

    # Om inget matchar
    return n.capitalize()

# Kör funktionen på din df_clean
df_clean['produktnamn'] = df_clean['produktnamn'].apply(clean_product)

In [6]:
#Städa säsonger
def clean_säsong(v):
    v = str(v).lower()
    året_runt = ['alla', 'hela året', 'all year', 'året runt']
    våren = ['vår', 'spring', 'v2024', 'april', ]
    for ord in året_runt:
        if ord in v:
            return 'Året runt'
    for ord in våren:
        if ord in v:
            return 'Våren'
    if 'höst' in v:
        return 'Hösten'
    if 'förodling' in v:
        return 'Förodling'
    return 'Okänd'
df_clean['säsong'] = df_clean['säsong'].apply(clean_säsong) 

In [7]:
#Städa zoner

def clean_zon(v):
    v = str(v).strip().lower()
    
    # Prioritera namn för presentationen
    if '1' in v or 'storstad' in v: return 'Storstad (Zon 1)'
    if '2' in v or 'södra' in v: return 'Södra Sverige (Zon 2)'
    if '3' in v or 'mellan' in v: return 'Mellansverige (Zon 3)'
    if '4' in v: return 'Längre norrut (Zon 4)'
    if '5' in v or 'norr' in v: return 'Norrland (Zon 5)'
    
    return 'Okänd zon'

df_clean['leveranszon'] = df_clean['leveranszon'].apply(clean_zon)



## 2.2 Hantering av saknade värden
Vi har identifierat och åtgärdat saknade värden (NaN) för att säkerställa att analysen inte blir missvisande.

* Alla null värde har jag sorterat till en egen "namn", exempelvis null värde på säsong ändrat till okänd, recensioner som inte har hanterat blir ingen recension etc

In [8]:
#Hantera nullvärden
#Fixa recensionerna (Text, Datum och Betyg)
df_clean['recension_text'] = df_clean['recension_text'].fillna('Ingen recension')
df_clean['recensionsdatum'] = df_clean['recensionsdatum'].fillna('Ej tillgängligt')
df_clean['betyg'] = df_clean['betyg'].fillna(0)

#Fixa önskat leveransdatum (Kopiera faktiskt datum)
df_clean['önskat_leveransdatum'] = df_clean['önskat_leveransdatum'].fillna(df_clean['faktiskt_leveransdatum'])

#Fixa säsong
df_clean['säsong'] = df_clean['säsong'].fillna('Okänd')

## 2.3 Hantering av tidsdata och beräkning av KPI
För att mäta logistikens effektivitet har vi konverterat datumkolumner till har vi skapat ny kolumn leveranstid_dagar

* Syfte: Att kunna identifiera logistiken i  per zon.

In [9]:
# Räkna ut dagarna
df_clean['leveranstid_dagar'] = (df_clean['faktiskt_leveransdatum'] - df_clean['orderdatum']).dt.days
# Gör alla dagar positiva
df_clean['leveranstid_dagar'] = df_clean['leveranstid_dagar'].abs()


## 2.4 Sentimentanalys av kundrecensioner med BERT
För att gå bortom den numeriska datan (betyg 1-5) och verkligen förstå kundernas upplevelse, använder vi en  NLP-modell

Vi använder BERT-modell som är tränad specifikt på recensioner

Vi mappar modellens 1-5 stjärnor till tre tydliga grupper:
* Negativ: (1-2 stjärnor)
* Neutral: (3 stjärnor)
* Positiv: (4-5 stjärnor)

Denna  gör att vi i analysen kan se om t.ex. sena leveranser direkt korrelerar med negativt sentiment i texten.

In [10]:
# Sentiment analys
from transformers import pipeline

sentiment_model = pipeline("sentiment-analysis", model="nlptown/bert-base-multilingual-uncased-sentiment")
def analyze_sentiment(text):
    # Om texten är "Ingen recension" eller tom, returnera direkt utan att köra AI:n
    if not text or text == 'Ingen recension' or str(text).lower() == 'nan':
        return 'Ingen recension'
    
    # Kör modellen bara på riktiga recensioner
    resultat = sentiment_model(str(text)[:510])[0]
    label = resultat['label']
    
    if label in ['1 star', '2 stars']:
        return 'Negativ'
    elif label == '3 stars':
        return 'Neutral'
    else:
        return 'Positiv'

# Kör analysen igen på din städade df_clean
df_clean['recensionssentiment'] = df_clean['recension_text'].apply(analyze_sentiment)


/Users/z/Documents/inlämning datascience/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 2230.19it/s, Materializing param=classifier.weight]                                      


## 2.5 Validering av sentimentmodell på extern data
För att säkerställa att vår BERT-modell (sentimentanalys) är tillförlitlig, testar vi den på en separat valideringsfil: gronagarden_validation.csv

**Process för validering:**
* Inläsning av extern valideringsdata.
* BERT-modellen klassificerar recensionerna i tre kategorier (Positiv, Neutral, Negativ).
* Ett stickprov på 15 rader genereras för att manuellt kontrollera modellens precision.

In [11]:
df_val = pd.read_csv('../gronagarden_validation.csv')

# Hantera nullvärden i valideringsdata
df_val['recension_text'] = df_val['recension_text'].fillna('Ingen recension')
df_val['säsong'] = df_val['säsong'].fillna('Okänd')

#Tvätta siffror
for col in ['vikt_gram', 'pris', 'antal']:
    if col in df_val.columns:
        df_val[col] = clean_siffra(df_val[col])

# Analysera sentiment
df_val['recensionssentiment'] = df_val['recension_text'].apply(analyze_sentiment)

# Spara till SQL
conn = sql.connect('gronagarden_data_cleaned.db')
df_val.to_sql('validation_results', conn, if_exists='replace', index=False)
conn.close()

# Visa resultatet
print("Validering klar!")
print(df_val[['recension_text', 'recensionssentiment']].sample(n=15))

Validering klar!
                                        recension_text recensionssentiment
86                                     Ingen recension     Ingen recension
406                                    Ingen recension     Ingen recension
32      Fröerna grodde på under en vecka, fantastiskt!             Positiv
351                                    Ingen recension     Ingen recension
392                                    Ingen recension     Ingen recension
288              Ok kvalitet men lite dyrt tycker jag.             Neutral
0                                      Ingen recension     Ingen recension
95   Jorden luktar fräscht och bra, plantorna älska...             Positiv
279                                    Ingen recension     Ingen recension
291           Halva fröerna var redan mögliga i påsen.             Negativ
334    Kom fram för sent, kunde lika gärna kastat det.             Neutral
1                  Skadad leverans, hälften obrukbart.             Negativ
269     

## 2.6 Slutlig lagring i SQL-databas
Nu när all data är tvättad, beräknad och analyserad, sparar vi resultaten i vår databas gronagarden_data_cleaned.db. 

**Vi skapar två tabeller:**
* orders: Innehåller den fullständiga städade försäljningsdatan från huvudfilen.
* validation_results: Innehåller resultaten från vår sentimentvalidering.

Genom att separera dessa kan vi i nästa steg (Fil 03) göra en ren KPI-analys på försäljningen samtidigt som vi kan utvärdera vår AI-modells träffsäkerhet.

In [12]:
df_clean = df_clean.map(lambda x: str(x) if isinstance(x, pd.Timestamp) else x)
# Anslut till databasen
conn = sql.connect('../gronagarden_data_cleaned.db')

# 2. Spara din df_clean till en tabell som heter 'orders'
df_clean.to_sql('orders', conn, if_exists='replace', index=False)

# 3. Stäng anslutningen
conn.close()
print("Datan är nu sparad i SQL databasen :)")

Datan är nu sparad i SQL databasen :)


## 2.7 Analys av kundnöjdhet per leveranszon
I detta steg undersöker vi hur kundernas upplevelse (sentiment) skiljer sig åt mellan de olika geografiska områdena. Genom att koppla samman vår BERT-analys med leveransdata kan vi se var butiken presterar bäst och var det finns utrymme för förbättring.



In [14]:
conn = sql.connect('../gronagarden_data_cleaned.db')
query = """
SELECT leveranszon, recensionssentiment, COUNT(*) as antal_recensioner
FROM gronagarden_orders
WHERE recensionssentiment != 'Ingen recension'
GROUP BY leveranszon, recensionssentiment
ORDER BY leveranszon, antal_recensioner DESC
"""
df_sentiment = pd.read_sql(query, conn)
conn.close()

# Visa resultatet
print(df_sentiment.head())

             leveranszon recensionssentiment  antal_recensioner
0  Längre norrut (Zon 4)             Positiv                 62
1  Längre norrut (Zon 4)             Negativ                 47
2  Längre norrut (Zon 4)             Neutral                 29
3  Mellansverige (Zon 3)             Positiv                116
4  Mellansverige (Zon 3)             Negativ                 69



## Dokumentation - ETL-Pipeline för Gröna Gårdens data

### Överblick
Denna pipeline transformerar rådatan från Gröna Gårdens försäljningssystem till ett rensat och analyserat dataset lagrat i SQL. Processen omfattar datatvätt, beräkning av KPI:er, sentimentanalys och validering.

### Steg i pipelinen

#### 1. **Datatvätt (Data Cleaning)**
- **clean_siffra()**: Konverterar textrepresentationer av tal (ex. "två") till numeriska värden
- **clean_datum()**: Standardiserar datumformat och hanterar svenska månadsnämnanden
- **clean_product()**: Grupperar produkter i logiska kategorier
- **clean_säsong()**: Normaliserar säsongsinformation
- **clean_zon()**: Konverterar leveranszoner till standardiserade namn

#### 2. **Hantering av saknade värden**
- Recensioner utan text: "Ingen recension"
- Betyg utan värde: 0
- Önskat leveransdatum: Kopieras från faktiskt leveransdatum
- Säsong: "Okänd"

#### 3. **KPI-beräkning**
- **leveranstid_dagar**: Beräknas från skillnaden mellan faktiskt och önskat leveransdatum (alltid positiva värden)

#### 4. **Sentimentanalys (BERT)**
Använder `nlptown/bert-base-multilingual-uncased-sentiment` modellen för att klassificera recensioner:
- **Negativ**: 1-2 stjärnor
- **Neutral**: 3 stjärnor
- **Positiv**: 4-5 stjärnor

#### 5. **Validering**
Extern valideringsfil (gronagarden_validation.csv) processas parallellt för att verifiera modellens träffsäkerhet.

#### 6. **Lagring**
Data sparas i `gronagarden_data_cleaned.db`:
- **Tabell: orders** - Huvuddatan från försäljningen
- **Tabell: validation_results** - Sentimentanalysresultat från validering

### Använd kolumner
| Kolumn | Typ | Beskrivning |
|--------|-----|-------------|
| produktnamn | string | Standardiserad produktnamn |
| orderdatum | datetime | Når order placerades |
| faktiskt_leveransdatum | datetime | Levererat datum |
| önskat_leveransdatum | datetime | Önska leveransdatum |
| leveranstid_dagar | int | Leveranstid i dagar |
| leveranszon | string | Geografisk leveranszon (1-5) |
| recension_text | string | Kundrecension |
| betyg | float | Kundbetyg (0-5) |
| recensionssentiment | string | AI-klassificering (Positiv/Neutral/Negativ) |
| säsong | string | Säsong när ordern gjordes |

### Output
-  Rensat dataset sparad i SQL
-  Sentimentanalys genomförd på alla recensioner
-  Validering utförd på extern data
-  KPI:er beräknade för logistikanalys

### Nästa steg
Se fil **03_kpi_analysis.ipynb** för:
- Detaljerad analys per leveranszon
- Korrelation mellan leveranstid och kundnöjdhet
- Visualisering av trender
